[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-IA_Ed2026/blob/main/meca-ia/C01/E01-1_Fleche_Poutre.ipynb)

# Flèche d'une poutre : retrouver une loi à partir d'exemples

## Le contexte

Un banc d'essai mesure la flèche de barres rectangulaires en flexion trois points : la barre repose sur deux appuis, une charge est appliquée en son milieu et un comparateur lit de combien elle fléchit. Chaque essai mobilise le banc, une éprouvette et du temps.

La résistance des matériaux donne une formule pour cette flèche. Ici, on fait comme si on l'ignorait : peut-on apprendre la flèche à partir de 480 essais déjà réalisés, puis la prédire pour des barres jamais essayées ?

C'est un problème de régression : la grandeur à prédire est un nombre, en millimètres.

Défi : prédire `fleche_mm` et obtenir le meilleur R² possible sur la base de test. La règle de découpage est fixée plus bas. La base de test sert à mesurer, jamais à apprendre ni à régler : si vous essayez dix variantes en regardant le test, il n'y a plus de test.

## Les données

Fichier `MecaIA-Fleche_Poutre.xlsx`, dans le dossier `donnees` du dépôt : 480 essais, 7 colonnes. Une ligne est un essai.

| Colonne | Signification | Unité |
|---|---|---|
| `materiau` | acier, titane ou aluminium | catégorie |
| `longueur_mm` | portée entre les deux appuis | mm |
| `largeur_mm` | largeur de la section | mm |
| `hauteur_mm` | hauteur de la section, dans le sens de la charge | mm |
| `charge_N` | charge appliquée au milieu | N |
| `temperature_C` | température ambiante du banc | °C |
| `fleche_mm` | cible : flèche mesurée au comparateur | mm |

Ce jeu est simulé : les flèches sont calculées par la loi de flexion des poutres (Euler-Bernoulli), puis perturbées par une dispersion du module d'un lot à l'autre et par un bruit de mesure. Il enseigne une méthode et ne remplace pas un essai. Quelques cases sont vides, comme sur un vrai relevé.

### Étape 1 : charger le fichier et regarder sa forme

On lit le classeur, puis on vérifie ses dimensions et ses cases vides avant toute modélisation. Le dossier des données est cherché depuis le dossier du notebook, puis depuis un clone du dépôt ; ouvert seul, par exemple dans Colab, le notebook télécharge une fois le jeu depuis le dépôt public.

In [ ]:
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import set_config

set_config(display="text")
GRAINE = 42
FICHIER = "MecaIA-Fleche_Poutre.xlsx"
URL_JEU = "https://raw.githubusercontent.com/philippe-amuzuga/Meca-IA_Ed2026/main/donnees/MecaIA-Fleche_Poutre.xlsx"

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-IA_Ed2026", "donnees")]
dossier = next((d for d in candidats if (d / FICHIER).is_file()), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique du jeu depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(URL_JEU, dossier / FICHIER)
    except OSError as erreur:
        raise FileNotFoundError(
            "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
df = pd.read_excel(dossier / FICHIER)
print("dimensions :", df.shape)
print("cases vides par colonne :")
print(df.isna().sum().to_string())
df.head()

Lecture du résultat : 480 lignes et 7 colonnes. Deux colonnes ont des cases vides, `largeur_mm` (8) et `temperature_C` (15). Un modèle linéaire refuse les cases vides : il faudra les remplir, sans tricher (étape 4).

### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas le dossier `donnees/` du dépôt. La cellule suivante ne télécharge rien : elle cherche ce dossier (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-IA_Ed2026`), vérifie la présence du classeur du défi et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez le dossier `donnees` à la racine de l'espace de travail et téléversez-y le classeur listé, pris dans le dépôt.

In [ ]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
from pathlib import Path

JEUX_LUS = ["MecaIA-Fleche_Poutre.xlsx"]                  # le classeur lu par ce notebook


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-IA_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees = chercher("donnees")
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Étape 2 : regarder la cible face aux variables

Avant de choisir un modèle, on trace la flèche en fonction de quatre variables numériques. Le but est de repérer les tendances et les variables sans effet.

In [ ]:
cible = "fleche_mm"
fig, axes = plt.subplots(2, 2, figsize=(8, 6))
for ax, colonne in zip(axes.ravel(), ["charge_N", "longueur_mm", "hauteur_mm", "temperature_C"]):
    ax.scatter(df[colonne], df[cible], s=12, alpha=0.6, color="0.25")
    ax.set_xlabel(colonne)
    ax.set_ylabel("flèche (mm)")
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

Lecture du résultat : la flèche croît avec la charge et, plus vite encore, avec la portée. Elle décroît quand la hauteur de la section augmente. Les nuages s'ouvrent en éventail : une même charge donne des flèches différentes, parce que le matériau et la section changent. La température ne montre aucune tendance.

### Étape 3 : séparer l'apprentissage et le test

Un quart des essais est mis de côté avant tout apprentissage. La graine 42 fixe le tirage : chacun obtient la même base de test, et les scores se comparent.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=cible)
y = df[cible]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=GRAINE)
print("apprentissage :", len(X_train), "essais")
print("test :", len(X_test), "essais")

### Étape 4 : construire le modèle simple

Le modèle de référence est volontairement simple. Il enchaîne trois gestes dans un pipeline : remplir les cases vides par la médiane de la colonne, transformer le matériau (du texte) en colonnes 0 ou 1 (encodage one-hot), puis ajuster une régression linéaire.

Le pipeline apprend sur `X_train` seul : les médianes sont calculées sur l'apprentissage, puis appliquées telles quelles au test. Rien du test ne fuit vers le modèle.

In [ ]:
# Votre code :

### Étape 5 : mesurer sur la base de test

Le modèle prédit les essais qu'il n'a jamais vus. Trois indicateurs sont lus : le coefficient de détermination R², et deux erreurs en millimètres, la racine de l'erreur quadratique moyenne (RMSE) et l'erreur absolue moyenne (MAE).

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

y_pred = modele.predict(X_test)
print(f"R² : {r2_score(y_test, y_pred):.3f}")
print(f"RMSE : {root_mean_squared_error(y_test, y_pred):.3f} mm")
print(f"MAE : {mean_absolute_error(y_test, y_pred):.3f} mm")
print(f"flèche moyenne de la base de test : {y_test.mean():.3f} mm")

Lecture du résultat : le R² vaut environ 0,72, soit près des trois quarts de la variation expliqués. L'erreur absolue moyenne est de 0,37 mm pour une flèche moyenne de 1,12 mm, donc d'environ un tiers : c'est trop pour une prédiction d'ingénieur.

### Étape 6 : regarder le résultat, pas seulement le score

On place la flèche prédite en fonction de la flèche mesurée. Une prédiction parfaite tomberait sur la diagonale.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(y_test, y_pred, s=18, color="0.25")
limites = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(limites, limites, color="black", lw=1, linestyle="--", label="prédiction parfaite")
ax.set_xlabel("flèche mesurée (mm)")
ax.set_ylabel("flèche prédite (mm)")
ax.legend(frameon=False)
ax.grid(alpha=0.3)
plt.show()
print("prédictions négatives :", int((y_pred < 0).sum()), "sur", len(y_pred))

Lecture du résultat : les points s'écartent de la diagonale, surtout pour les grandes flèches. Le modèle prédit même des flèches négatives, ce qui n'a aucun sens physique. Le défaut vient de la forme du modèle, une droite, qui ne peut pas épouser une loi où la flèche varie comme une puissance de la portée. Ce n'est pas un manque de données.

## Pour aller plus loin : donner la forme de la loi au modèle

Le modèle simple de l'étape 4 reste le score de référence du défi. La cellule suivante explore une piste pour faire mieux : changer la manière dont les variables entrent dans le modèle, et lire ce que le modèle a appris. Comparez les exposants affichés à la formule de la flèche d'une poutre sur deux appuis chargée en son milieu.

In [ ]:
# Votre code :

Lecture du résultat : la cellule affiche le R² du modèle sur les logarithmes, les exposants appris et leur lecture physique. Ce R² n'est pas le score publié du défi.

## Le score de référence

Le score publié est le R² du modèle simple de l'étape 4 sur la base de test. L'empreinte identifie cette base de test : si la vôtre diffère, le découpage n'est pas le même et les scores ne se comparent pas.

In [ ]:
import hashlib

positions = ",".join(str(i) for i in sorted(X_test.index))
print("EMPREINTE_TEST =", hashlib.sha256(positions.encode("ascii")).hexdigest())
print("SCORE_REFERENCE r2 = %.3f" % r2_score(y_test, modele.predict(X_test)))

## Synthèse

1. Un modèle apprend sur 360 essais et se juge sur 120 essais qu'il n'a jamais vus.
2. Le pipeline calcule ses médianes sur l'apprentissage seul : c'est ce qui évite la fuite de données.
3. Le R² de référence est le score à battre. Une droite sur les variables brutes ne représente pas une loi en puissance.
4. Connaître la physique aide à choisir la forme du modèle : un produit de puissances devient une somme avec les logarithmes.
5. Un score s'annonce avec une erreur en unité physique, ici des millimètres, pas seulement avec un R².